In [0]:
Path = "/Workspace/Users/anidesifitriaaaa@gmail.com/Drafts/Databricks/gold_layer_data"

df_ml = spark.read.format("delta").load(Path)

df_ml.show(10, False)

df_ml.printSchema()


In [0]:
df_ml.isEmpty()


In [0]:
from pyspark.sql.functions import col

df_ml = df_ml.drop("Patient_ID", "filename", "ingesttime", "Room_Number", "Doctor", "Date_of_Admission", "Discharge_Date", "Hospital").withColumn("Billing_Amount", col("Billing_Amount").cast("double"))
 
df_ml.show(10, False)

In [0]:
print(df_ml.dtypes)

In [0]:
train_spark, test_spark = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Training Dataset Count: {train_spark.count()}")
print(f"Testing Dataset Count: {test_spark.count()}")

In [0]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import DecisionTreeRegressor

#string indexer
categorical_columns = [field for (field, dataType) in df_ml.dtypes if dataType == "string"]

OutputIndex_cols = [x + "Index" for x in categorical_columns]

string_indexer = StringIndexer(inputCols=categorical_columns, outputCols=OutputIndex_cols, handleInvalid="skip")

numerical_columns = [field for (field, dataType) in df_ml.dtypes if dataType in ("double", "int") and field != "Estimated_Length_of_Stay"]

VectorAssemblerInput_cols = OutputIndex_cols + numerical_columns

vector_assembler = VectorAssembler(inputCols=VectorAssemblerInput_cols, outputCol="features")

print("Preprocessing has been completed")

In [0]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(featuresCol="features", labelCol="Estimated_Length_of_Stay")

pipeline = Pipeline(stages=[string_indexer, vector_assembler, dt])

pipeline_train = pipeline.fit(train_spark)

pred_dt = pipeline_train.transform(test_spark)

pred_dt.select("features", "Estimated_Length_of_Stay","prediction").show(10, False)


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(labelCol="Estimated_Length_of_Stay", predictionCol="prediction", metricName="mae")

mae = evaluator.evaluate(pred_dt)

print(f"MAE is {mae:.2f}")

r2_dt = evaluator.setMetricName("r2").evaluate(pred_dt)
print(f"R2 is {r2_dt}")

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

tree_model_dt = pipeline_train.stages[-1]

feature_importances = tree_model_dt.featureImportances.toArray()

list_feature = VectorAssemblerInput_cols

df_fi = pd.DataFrame({"Feature": list_feature, "Importance": feature_importances})

df_fi.sort_values(by="Importance", ascending=False) 

#Visualization
plt.figure(figsize=(18, 6))
sns.barplot(x="Feature", y="Importance", data=df_fi, palette='viridis'
            )
plt.ylabel('importance_score', fontsize = 12)
plt.xlabel("List Feature", fontsize = 12)
plt.title("Feature Importance for Decision Tree", fontsize = 14, fontweight="bold")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()


In [0]:
plt.figure(figsize=(20, 6))

df_plot = pred_dt.select("Estimated_Length_of_Stay", "prediction").toPandas()

# error (Residual)
df_plot['Residuals'] = df_plot['Estimated_Length_of_Stay'] - df_plot['prediction']

# scatter plot
sns.scatterplot(x='Estimated_Length_of_Stay', y='prediction', data= df_plot, alpha=0.4, color='blue')

# red digonal line
min_val = df_plot['Estimated_Length_of_Stay'].min()
max_val = df_plot['Estimated_Length_of_Stay'].max()

plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)

#labelling
plt.title('Actual vs Predicted: Estimated Length of Stay', fontsize=14, fontweight='bold')
plt.xlabel('Actual (Original Day of Stay)', fontsize=12)
plt.ylabel('Prediction (Model predicted Day of Stay)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(['Actual vs Predicted', 'Ideal Line'], loc='upper left', fontsize=12)

plt.show()

In [0]:
plt.figure(figsize=(20, 6))

# error (Residuals)
sns.histplot(df_plot['Residuals'], bins=40, kde=True, color='purple')

# Line reference on error is 0
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)

plt.title('Distribution of Prediction Errors (Residuals)', fontsize=14, fontweight='bold')
plt.xlabel('Error (Day)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

plt.show()

In [0]:
df_ml = df_ml.drop("Test_Results", "Gender", "Admission_Type")
df_ml.show(10, False)

In [0]:
train_spark_lr, test_spark_lr = df_ml.randomSplit([0.8, 0.2], seed=42)

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

from pyspark.ml.regression import LinearRegression 

# List categorical & Numerical Column
cat_cols_lr = [field for (field, dataType) in df_ml.dtypes if dataType == "string"]
num_cols_lr = [field for (field, dataType) in df_ml.dtypes if dataType in ("double", "int") and field != "Estimated_Length_of_Stay"]

# 2. Categorical (String Indexer -> OHE)
index_cols_lr = [x + "_IndexLR" for x in cat_cols_lr]
indexer_lr = StringIndexer(inputCols=cat_cols_lr, outputCols=index_cols_lr, handleInvalid="skip")

ohe_cols_lr = [x + "_OHE" for x in cat_cols_lr]
ohe_encoder_lr = OneHotEncoder(inputCols=index_cols_lr, outputCols=ohe_cols_lr)

# 3. Numerical (VectorAssembler Khusus Numerik -> StandardScaler)
# StandardScaler needs to transform the data into vector
assembler_num_lr = VectorAssembler(inputCols=num_cols_lr, outputCol="num_vector_raw")
scaler_lr = StandardScaler(inputCol="num_vector_raw", outputCol="num_vector_scaled")

#Final Assembler (Gather the OHE & Numerical)
assembler_inputs_lr = ohe_cols_lr + ["num_vector_scaled"]
final_assembler_lr = VectorAssembler(inputCols=assembler_inputs_lr, outputCol="features_lr")

#lr
lr = LinearRegression(featuresCol="features_lr", labelCol="Estimated_Length_of_Stay")

pipeline_lr = Pipeline(stages=[indexer_lr, ohe_encoder_lr, assembler_num_lr, scaler_lr, final_assembler_lr, lr])

# Fitting to LR
pipeline_model_lr = pipeline_lr.fit(train_spark_lr) 

# Validation & Testing
pred_lr = pipeline_model_lr.transform(test_spark_lr)

df_lr = pred_lr.select("features_lr", "Estimated_Length_of_Stay", "prediction")

df_lr.show(10, False)

In [0]:
#Evaluating model uses MAE, R2
from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(
    labelCol="Estimated_Length_of_Stay",
    predictionCol="prediction",
    metricName="mae")

rmse = evaluator.evaluate(pred_lr)
print(f"MAE is {mae:.2f}")

r2_lr = evaluator.setMetricName("r2").evaluate(pred_lr)
print(f"R2 is {r2_lr}")

In [0]:
from matplotlib import pyplot as plt
import seaborn as sns

plt.figure(figsize=(20, 6))

# Actual vs Predicted
sns.scatterplot(x='Estimated_Length_of_Stay', y='prediction', data=df_lr.toPandas(), color='blue', alpha=0.5)

# Red Diagonal Line
min_val = df_plot['Estimated_Length_of_Stay'].min()
max_val = df_plot['Estimated_Length_of_Stay'].max()
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)

plt.title('Actual vs Predicted Length of Stay')
plt.xlabel('Actual Length of Stay')
plt.ylabel('Predicted Length of Stay')
plt.show()